# CASE 01 — Square 6×6 + 3×3 — Correlation TopLeft

In [1]:
%%writefile case01_sq_3x3_corrTopLeft.cu

#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

#define WIDTH       6
#define HEIGHT      6
#define MASK_WIDTH  3
#define MASK_HEIGHT 3
#define BLOCK_SIZE  8

__global__ void correlationTopLeft(int *dA, int *dMask, int *dC,
                                   int width, int height,
                                   int mWidth, int mHeight)
{
    int col = threadIdx.x + blockIdx.x * blockDim.x;
    int row = threadIdx.y + blockIdx.y * blockDim.y;
    if (row < height && col < width)
    {
        int sum = 0;
        for (int i = 0; i < mHeight; i++)
            for (int j = 0; j < mWidth; j++)
            {
                int r = row + i;        /* top-left: no offset */
                int c = col + j;
                if (r < height && c < width)
                    sum += dA[r*width+c] * dMask[i*mWidth+j];
            }
        dC[row*width+col] = sum;
    }
}

void printMatrix(const char *label, int *M, int w, int h)
{
    printf("\n%s:\n", label);
    for (int r = 0; r < h; r++)
    {
        for (int c = 0; c < w; c++)
            printf("%6d", M[r*w+c]);
        printf("\n");
    }
}

int main()
{
    int size     = WIDTH * HEIGHT * sizeof(int);
    int maskSize = MASK_WIDTH * MASK_HEIGHT * sizeof(int);

    int *hA    = (int*) malloc(size);
    int *hMask = (int*) malloc(maskSize);
    int *hC    = (int*) malloc(size);

    srand(time(NULL));
    for (int i = 0; i < WIDTH*HEIGHT; i++)
        hA[i] = rand()%9+1;

    int tempMask[3][3] = {{1,0,-1},{2,0,-2},{1,0,-1}};
    for (int i = 0; i < MASK_HEIGHT; i++)
        for (int j = 0; j < MASK_WIDTH; j++)
            hMask[i*MASK_WIDTH+j] = tempMask[i][j];

    printMatrix("Input Matrix (6x6)", hA,    WIDTH,      HEIGHT);
    printMatrix("Mask (3x3)",         hMask, MASK_WIDTH, MASK_HEIGHT);

    int *dA, *dMask, *dC;
    cudaMalloc((void**)&dA,    size);
    cudaMalloc((void**)&dMask, maskSize);
    cudaMalloc((void**)&dC,    size);
    cudaMemcpy(dA,    hA,    size,     cudaMemcpyHostToDevice);
    cudaMemcpy(dMask, hMask, maskSize, cudaMemcpyHostToDevice);

    dim3 DimBlock(BLOCK_SIZE, BLOCK_SIZE, 1);
    dim3 DimGrid((int)ceil((float)WIDTH/BLOCK_SIZE),
                 (int)ceil((float)HEIGHT/BLOCK_SIZE), 1);

    cudaEvent_t start, stop; float gpuTime;
    cudaEventCreate(&start); cudaEventCreate(&stop);
    cudaEventRecord(start);

    correlationTopLeft<<<DimGrid,DimBlock>>>(dA,dMask,dC,
                        WIDTH,HEIGHT,MASK_WIDTH,MASK_HEIGHT);

    cudaEventRecord(stop); cudaEventSynchronize(stop);
    cudaEventElapsedTime(&gpuTime, start, stop);
    cudaMemcpy(hC, dC, size, cudaMemcpyDeviceToHost);

    printMatrix("OUTPUT: Correlation TopLeft (6x6, 3x3)", hC, WIDTH, HEIGHT);
    printf("\nGPU Time: %.4f ms\n", gpuTime);
    printf("Grid: %dx%d  Block: %dx%d\n",
            DimGrid.x,DimGrid.y,DimBlock.x,DimBlock.y);

    cudaFree(dA); cudaFree(dMask); cudaFree(dC);
    free(hA); free(hMask); free(hC);
    cudaEventDestroy(start); cudaEventDestroy(stop);
    return 0;
}

Writing case01_sq_3x3_corrTopLeft.cu


In [2]:
!nvcc -arch=sm_75 case01_sq_3x3_corrTopLeft.cu -o case01_sq_3x3_corrTopLeft

!./case01_sq_3x3_corrTopLeft


Input Matrix (6x6):
     8     6     9     1     2     5
     8     2     9     2     3     2
     3     6     8     2     3     8
     4     5     2     1     9     7
     3     3     7     9     6     4
     8     3     7     5     3     6

Mask (3x3):
     1     0    -1
     2     0    -2
     1     0    -1

OUTPUT: Correlation TopLeft (6x6, 3x3):
    -8     9    24   -10    11    17
    -9    12     9   -18    18    25
    -5     6    -8   -13    27    26
    -5   -10    -1     3    24    21
    -2   -10     9     3    12    16
     1    -2     4    -1     3     6

GPU Time: 0.1147 ms
Grid: 1x1  Block: 8x8
